# Fine-tuning de mBERT frente al baseline de embeddings congelados (complemento del OE4)

**Pregunta:** ¿reentrenar mBERT con las oraciones reales del corpus mejora al baseline del OE2 (mBERT congelado + Regresión Logística, F1 macro 0.665 con el protocolo B)?

**Es un experimento exploratorio y separado.** No cambia ningún resultado del OE2 ni del OE3: usa los mismos datos y las mismas particiones, y se compara con las mismas reglas.

| Elemento | Igual que en el OE2 | Diferente (a propósito) |
|---|---|---|
| Corpus | 700 oraciones, 7 intenciones, texto normalizado (minúsculas, sin puntuación, sin tildes ni ñ) | |
| Particiones | Las 5 particiones congeladas (`folds_fijos.csv`) | |
| Oraciones de prueba | Nunca se usan para elegir nada | |
| Modelo | `bert-base-multilingual-cased` | **Se reentrena** (antes estaba congelado) |
| Clasificador | | Capa de clasificación entrenada junto con el modelo (en vez de Regresión Logística) |
| Hiperparámetros | Se eligen **sin tocar la prueba** (análogo al protocolo B) | Tasa de aprendizaje y épocas, elegidas con validación interna del entrenamiento de cada partición |
| Repeticiones | | 3 semillas: el fine-tuning varía entre corridas |
| Comparación | F1 macro agrupado de las 700 predicciones, IC 95% bootstrap y **diferencia pareada** contra el baseline | |

**Variantes:** `completo` (se actualizan todos los pesos de mBERT) y, opcionalmente, `lora` (adaptadores LoRA, menos parámetros; suele ser más estable con pocos datos).

**Qué NO hace este cuaderno:** no usa la API de Claude (no gasta dinero) y no agrega texto sintético. Solo datos reales.

**Tiempo estimado en una GPU T4 de Colab:** unos 60 minutos para la variante `completo` con 3 semillas y 5 particiones. Todo se guarda en tu Drive a medida que avanza: si Colab se desconecta, vuelve a correr desde la celda de configuración y retoma.

**Uso:** Entorno de ejecución → *Cambiar tipo de entorno de ejecución* → GPU (T4). Corre las celdas en orden. Si quieres comprobar que todo funciona antes, pon `PRUEBA = True` en la celda de configuración (unos minutos).

## 0. Montar Google Drive (aquí queda guardado el avance y los resultados)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clonar tu repositorio

In [ ]:
import os
GITHUB_USUARIO = "LhiaHC"

%cd /content
!rm -rf /content/shiwilu-tesis
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git log --oneline -3

REPO = '/content/shiwilu-tesis'
requeridos = [
    f'{REPO}/corpus/corpus_shiwilu_final.csv',
    f'{REPO}/oe2_aumento_de_datos/particiones/folds_fijos.csv',
    f'{REPO}/oe2_aumento_de_datos/evaluacion/resultados/cv_interna/validacion_cruzada_predicciones_sin_puntuacion_cv_interna_c120.csv',
    f'{REPO}/shiwilu/clasificacion.py',
]
faltan = [r for r in requeridos if not os.path.exists(r)]
assert not faltan, 'Faltan archivos en el clon (haz git push de lo nuevo y vuelve a correr esta celda): ' + ', '.join(faltan)
print('OK')

## 2. Instalar dependencias y comprobar la GPU

`torchao` se desinstala a propósito: Colab trae una versión vieja que choca con `peft` (error «Found an incompatible version of torchao») y este cuaderno no la usa.
Si cambias de versión de Colab y ese error reaparece, corre solo esta celda otra vez y luego **Entorno de ejecución → Reiniciar sesión** (no pierdes nada: el avance está en Drive).

In [ ]:
!pip install -q transformers peft accelerate scikit-learn
!pip uninstall -y -q torchao
import torch
print('GPU disponible:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU (cambia el tipo de entorno de ejecucion)')

## 3. Configuración (corre esta celda en cada sesión)

- `VARIANTES`: `["completo"]` es el alcance mínimo. Agrega `"lora"` para probar también LoRA (suma ~60 minutos más).
- `SEMILLAS`: el fine-tuning no da siempre el mismo resultado; se repite con varias semillas y se promedia.
- `LRS` y `EPOCAS`: rejilla de hiperparámetros. En cada partición y semilla se elige la combinación con mejor F1 macro en **validación interna** (oraciones del entrenamiento apartadas; **nunca** la prueba).
- `N_INTERNOS`: con cuántas particiones internas se promedia esa elección (más = elección menos ruidosa, pero más lento). Se usa 2 para no repetir el problema del protocolo A (elegir con muy pocas oraciones).
- `PRUEBA = True`: corrida mínima para comprobar que todo funciona (1 partición, 1 semilla, 1 combinación, 2 épocas).

In [ ]:
import os, re, sys, math, time, random, glob, shutil
import numpy as np, pandas as pd, torch

VARIANTES = ["completo"]                 # alcance minimo; agrega "lora" si quieres
SEMILLAS = [0, 1, 2]
FOLDS = [0, 1, 2, 3, 4]
MODELO = "bert-base-multilingual-cased"
LRS = {"completo": [2e-5, 5e-5, 1e-4], "lora": [1e-4, 3e-4, 1e-3]}
EPOCAS = [5, 10]
N_INTERNOS = 2
BATCH = 16
MAX_LEN = 64
REPETICIONES_BOOTSTRAP = 2000
PRUEBA = False                           # True = comprobacion rapida

if PRUEBA:
    VARIANTES, SEMILLAS, FOLDS, LRS, EPOCAS, N_INTERNOS = ["completo"], [0], [0], {"completo": [5e-5]}, [2], 1

SUFIJO = "_prueba" if PRUEBA else ""
DIR_DRIVE = "/content/drive/MyDrive/shiwilu_finetuning" + SUFIJO       # avance y resultados (sobrevive a desconexiones)
DIR_RUNS = DIR_DRIVE + "/corridas"
os.makedirs(DIR_RUNS, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
USAR_FP16 = DEV == "cuda"
print("Dispositivo:", DEV, "| variantes:", VARIANTES, "| semillas:", SEMILLAS, "| particiones:", FOLDS)

## 4. Cargar los datos (los mismos del OE2)

Se usa exactamente el mismo corpus, el mismo texto normalizado y las mismas particiones congeladas que en los 12 experimentos. `pos` es la posición de cada oración en el corpus (0 a 699), igual que en los archivos de predicciones del OE2, lo que permite la comparación pareada.

In [ ]:
sys.path.insert(0, REPO)
from shiwilu.clasificacion import cargar_corpus, cargar_corpus_normalizado, cargar_folds, _normalizar_estricto
from shiwilu.taxonomia import INTENCIONES

CLASES = sorted(INTENCIONES)                                  # AFI, DES, EMO, NEG, PRG, REQUEST, SAL
corpus = cargar_corpus()
assert corpus.index.equals(pd.RangeIndex(len(corpus)))
texto = cargar_corpus_normalizado()["shiwilu"].astype(str).to_numpy()
y = corpus["intencion"].to_numpy()
Y_IDX = np.array([CLASES.index(c) for c in y])
folds = cargar_folds(corpus)
grupos = np.array([_normalizar_estricto(t) for t in corpus["shiwilu"]])
print(len(corpus), "oraciones |", len(CLASES), "intenciones | tamanos de prueba por particion:", [int((folds == f).sum()) for f in range(5)])
print("Ejemplos:", list(texto[:3]))

# --- funciones de medida (F1 macro rapido, igual al de sklearn con zero_division=0)
def f1_macro_idx(yv, pv, K=len(CLASES)):
    cm = np.bincount(yv * K + pv, minlength=K * K).reshape(K, K)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn
    return float(np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0).mean())

def f1_por_clase(yv, pv, K=len(CLASES)):
    cm = np.bincount(yv * K + pv, minlength=K * K).reshape(K, K)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)

## 5. Funciones de entrenamiento

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedGroupKFold

tok = AutoTokenizer.from_pretrained(MODELO)
enc = tok(list(texto), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
IDS, MASK = enc["input_ids"], enc["attention_mask"]
print("Ancho de secuencia (piezas):", IDS.shape[1])

def fijar_semilla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def entrenar_y_predecir(idx_tr, idx_evals, lr, epocas, semilla, variante):
    """Entrena con las oraciones idx_tr y devuelve las predicciones (indices de clase) de cada conjunto de idx_evals."""
    fijar_semilla(semilla)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODELO, num_labels=len(CLASES), id2label=dict(enumerate(CLASES)), label2id={c: i for i, c in enumerate(CLASES)})
    if variante == "lora":
        from peft import LoraConfig, get_peft_model
        model = get_peft_model(model, LoraConfig(task_type="SEQ_CLS", r=16, lora_alpha=32, lora_dropout=0.1, target_modules=["query", "value"]))
    model.to(DEV)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    pasos = epocas * math.ceil(len(idx_tr) / BATCH)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * pasos), pasos)
    scaler = torch.amp.GradScaler("cuda", enabled=USAR_FP16)
    g = torch.Generator().manual_seed(semilla)
    ytr = torch.as_tensor(Y_IDX[idx_tr])
    model.train()
    for _ in range(epocas):
        perm = torch.randperm(len(idx_tr), generator=g)
        for k in range(0, len(perm), BATCH):
            b = perm[k:k + BATCH]
            ii = torch.as_tensor(idx_tr[b.numpy()])
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USAR_FP16):
                salida = model(input_ids=IDS[ii].to(DEV), attention_mask=MASK[ii].to(DEV), labels=ytr[b].to(DEV))
            opt.zero_grad(set_to_none=True)
            scaler.scale(salida.loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sch.step()
    model.eval(); salidas = []
    with torch.no_grad():
        for idx in idx_evals:
            pr = []
            for k in range(0, len(idx), 64):
                ii = torch.as_tensor(idx[k:k + 64])
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USAR_FP16):
                    lg = model(input_ids=IDS[ii].to(DEV), attention_mask=MASK[ii].to(DEV)).logits
                pr.append(lg.argmax(-1).cpu().numpy())
            salidas.append(np.concatenate(pr))
    del model, opt
    if DEV == "cuda":
        torch.cuda.empty_cache()
    return salidas

def correr_particion(f, semilla, variante):
    """Elige (lr, epocas) con validacion interna del entrenamiento y predice la prueba de la particion f."""
    idx_test = np.where(folds == f)[0]
    idx_pool = np.where(folds != f)[0]
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=semilla)
    internos = list(cv.split(idx_pool, Y_IDX[idx_pool], groups=grupos[idx_pool]))[:N_INTERNOS]
    candidatos = []
    for lr in LRS[variante]:
        for ep in EPOCAS:
            f1s = []
            for tr, va in internos:
                p = entrenar_y_predecir(idx_pool[tr], [idx_pool[va]], lr, ep, semilla, variante)[0]
                f1s.append(f1_macro_idx(Y_IDX[idx_pool[va]], p))
            candidatos.append({"lr": lr, "epocas": ep, "f1_validacion_interna": float(np.mean(f1s))})
    cand = pd.DataFrame(candidatos)
    mejor = max(candidatos, key=lambda c: (c["f1_validacion_interna"], -c["epocas"], -c["lr"]))   # desempate: menos epocas, menor lr
    pred = entrenar_y_predecir(idx_pool, [idx_test], mejor["lr"], mejor["epocas"], semilla, variante)[0]
    salida = pd.DataFrame({"variante": variante, "semilla": semilla, "fold": f, "pos": idx_test, "real": y[idx_test],
                           "prediccion": [CLASES[i] for i in pred], "lr": mejor["lr"], "epocas": mejor["epocas"],
                           "f1_validacion_interna": mejor["f1_validacion_interna"]})
    return salida, cand

## 6. Correr los experimentos (retoma sola si Colab se desconecta)

Para cada variante, semilla y partición se guarda un archivo en tu Drive en cuanto termina. Si vuelves a correr la celda, salta lo que ya existe.
Cada partición imprime la combinación elegida y su F1 de validación interna (no es el F1 de prueba).

In [ ]:
t_ini = time.time()
for variante in VARIANTES:
    for semilla in SEMILLAS:
        for f in FOLDS:
            ruta = f"{DIR_RUNS}/{variante}_s{semilla}_f{f}.csv"
            if os.path.exists(ruta):
                print(f"[omitido] {variante} semilla {semilla} particion {f}")
                continue
            t0 = time.time()
            salida, cand = correr_particion(f, semilla, variante)
            cand.to_csv(f"{DIR_RUNS}/seleccion_{variante}_s{semilla}_f{f}.csv", index=False)
            salida.to_csv(ruta, index=False)
            f1_prueba = f1_macro_idx(Y_IDX[salida.pos.to_numpy()], np.array([CLASES.index(c) for c in salida.prediccion]))
            print(f"{variante} semilla {semilla} particion {f}: elegido lr={salida.lr.iloc[0]:g}, epocas={salida.epocas.iloc[0]}, "
                  f"F1 val. interna={salida.f1_validacion_interna.iloc[0]:.3f} | F1 prueba de esta particion={f1_prueba:.3f} | {time.time() - t0:.0f} s")
print(f"\nTiempo total de esta sesion: {(time.time() - t_ini) / 60:.1f} min")

## 7. Resultados y comparación pareada contra el baseline

Se unen las 5 particiones de cada semilla (700 predicciones) y se calcula el **F1 macro agrupado**, igual que en el OE2. La comparación con el baseline (mBERT congelado, sin aumento, protocolo B) es **pareada**: se remuestrean las mismas oraciones y se promedia el F1 de las semillas.

In [ ]:
base = pd.read_csv(f"{REPO}/oe2_aumento_de_datos/evaluacion/resultados/cv_interna/validacion_cruzada_predicciones_sin_puntuacion_cv_interna_c120.csv")
base = base[(base.modelo == "mbert") & (base.tecnica == "sin_aumento")].sort_values("pos")
assert len(base) == 700 and (base.real.to_numpy() == y).all(), "el baseline no esta alineado con el corpus"
P_BASE = np.array([CLASES.index(c) for c in base.prediccion])
F1_BASE = f1_macro_idx(Y_IDX, P_BASE)
print(f"Baseline OE2 (mBERT congelado + Regresion Logistica, protocolo B): F1 macro = {F1_BASE:.3f}\n")

def cargar_variante(variante):
    # devuelve {semilla: (posiciones, predicciones)}; exige las 700 oraciones salvo en modo PRUEBA
    preds = {}
    for semilla in SEMILLAS:
        partes = [pd.read_csv(r) for f in FOLDS for r in [f"{DIR_RUNS}/{variante}_s{semilla}_f{f}.csv"] if os.path.exists(r)]
        if not partes:
            continue
        d = pd.concat(partes).sort_values("pos")
        if len(d) == 700 or PRUEBA:
            preds[semilla] = (d.pos.to_numpy(), np.array([CLASES.index(c) for c in d.prediccion]))
    return preds

rng = np.random.default_rng(42)
filas, por_clase = [], {}
for variante in VARIANTES:
    preds = cargar_variante(variante)
    if not preds:
        print(f"[{variante}] aun no hay las 700 predicciones de ninguna semilla; corre el paso 6 completo.")
        continue
    pos = next(iter(preds.values()))[0]                      # oraciones comunes (las 700; en PRUEBA solo la particion 0)
    yv, pb = Y_IDX[pos], P_BASE[pos]
    P = np.stack([p for (_, p) in preds.values()])
    f1s = {s: f1_macro_idx(yv, p) for s, (_, p) in preds.items()}
    f1_base_sub = f1_macro_idx(yv, pb)
    n = len(yv); difs, medias = [], []
    for _ in range(REPETICIONES_BOOTSTRAP):
        i = rng.integers(0, n, n)
        f1_ft = np.mean([f1_macro_idx(yv[i], p[i]) for p in P])
        difs.append(f1_ft - f1_macro_idx(yv[i], pb[i])); medias.append(f1_ft)
    dif = float(np.mean(list(f1s.values())) - f1_base_sub)
    lo, hi = np.percentile(difs, [2.5, 97.5])
    filas.append({"variante": variante, "oraciones": n, "semillas": len(preds), "f1_baseline": f1_base_sub, "f1_media": np.mean(list(f1s.values())),
                  "f1_sd_entre_semillas": np.std(list(f1s.values())), "f1_minimo": min(f1s.values()), "f1_maximo": max(f1s.values()),
                  "ic95_bajo": np.percentile(medias, 2.5), "ic95_alto": np.percentile(medias, 97.5),
                  "dif_vs_baseline": dif, "dif_ic95_bajo": lo, "dif_ic95_alto": hi, "distinguible_de_cero": bool(lo > 0 or hi < 0)})
    por_clase.setdefault("baseline (congelado)", f1_por_clase(yv, pb))
    por_clase[f"fine-tuning {variante}"] = np.mean([f1_por_clase(yv, p) for p in P], axis=0)
RES = pd.DataFrame(filas)
pd.set_option("display.width", 220)
print(RES.round(3).to_string(index=False))
print("\nF1 por intencion (promedio de semillas):")
print(pd.DataFrame(por_clase, index=CLASES).round(3).to_string())

## 8. Guardar y descargar los resultados

Los CSV se guardan en tu Drive y en una carpeta `finetuning_resultados.zip` que se descarga. Para llevarlos al repositorio, copia el contenido a `oe4_sintesis/finetuning/resultados/` (sin hacer commit hasta revisarlo).

In [ ]:
DIR_OUT = f"{DIR_DRIVE}/resultados"
os.makedirs(DIR_OUT, exist_ok=True)
if len(RES):
    RES.to_csv(f"{DIR_OUT}/finetuning_vs_baseline.csv", index=False)
    pd.DataFrame(por_clase, index=CLASES).to_csv(f"{DIR_OUT}/f1_por_intencion.csv")
for variante in VARIANTES:
    partes = [pd.read_csv(r) for r in sorted(glob.glob(f"{DIR_RUNS}/{variante}_s*_f*.csv"))]
    if partes:
        pd.concat(partes).to_csv(f"{DIR_OUT}/predicciones_{variante}.csv", index=False)
    sel = [pd.read_csv(r).assign(archivo=os.path.basename(r)) for r in sorted(glob.glob(f"{DIR_RUNS}/seleccion_{variante}_*.csv"))]
    if sel:
        pd.concat(sel).to_csv(f"{DIR_OUT}/seleccion_hiperparametros_{variante}.csv", index=False)
shutil.make_archive("/content/finetuning_resultados", "zip", DIR_OUT)
print("Guardado en:", DIR_OUT, "|", sorted(os.listdir(DIR_OUT)))
try:
    from google.colab import files
    files.download("/content/finetuning_resultados.zip")
except Exception as e:
    print("Descarga manual desde Drive:", e)

## Cómo leer el resultado (y qué decir en el informe)

- **`dif_vs_baseline`** es el F1 macro del fine-tuning (promedio de semillas) menos el del baseline (0.665). Es *distinguible de cero* solo si su intervalo de confianza **no incluye el 0**.
- **Si mejora de forma distinguible:** reentrenar mBERT supera a los embeddings congelados con este corpus. Se reporta como complemento del OE4, y conviene repetir la comparación con LoRA.
- **Si no se distingue:** no hay evidencia de que reentrenar ayude con ~560 oraciones; es un resultado útil que respalda el diseño con embeddings congelados.
- **Si empeora:** con tan pocos datos el reentrenamiento puede sobreajustar. También es un resultado válido.
- **Mira `f1_sd_entre_semillas`:** si es grande (por ejemplo más de 0.02), el fine-tuning es inestable y el promedio debe leerse con cautela.
- **Mira `f1_por_intencion.csv`:** en particular DES, la intención más débil del baseline (0.54).
- Las diferencias menores a unos 0.01 no se interpretan, como en el resto de la tesis.

**Lo que este experimento no concluye:** que el fine-tuning «entienda» el shiwilu (el clasificador de n-gramas ya alcanza 0.751), ni que valga para otras lenguas. Solo compara dos formas de usar mBERT sobre este corpus.